## 环境准备：加载 Qwen 大模型

本 Notebook 使用 **ModelScope** 加载 **Qwen2.5-7B-Instruct** 模型，
替代原有的 MockLLM / 模拟 LLM，实现真实的模型推理。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

In [ ]:
# ============================================================
# 安装依赖（如需要，取消注释后运行）
# ============================================================
# !pip install modelscope transformers torch -q

# ============================================================
# QwenLLM 封装类：基于 ModelScope 加载 Qwen2.5 模型
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch


class QwenLLM:
    """
    基于 ModelScope 的 Qwen2.5 大模型封装类

    支持：
    - system prompt 设置
    - 多轮对话上下文维护
    - GPU / CPU 自动检测
    - 温度与生成长度控制
    """

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        """
        初始化 Qwen 模型

        Args:
            model_name: 模型 ID，默认 7B；低显存可改为 "Qwen/Qwen2.5-3B-Instruct"
            device: 指定设备，None 表示自动检测
        """
        # GPU / CPU 自动检测
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")
        print(f"[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct")

        # 加载模型和分词器
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 多轮对话历史
        self.messages = []

        print(f"[QwenLLM] 模型加载完成")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """
        对话接口

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            max_new_tokens: 最大生成 token 数
            temperature: 采样温度

        Returns:
            模型生成的回复文本
        """
        # 构建消息列表
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # 更新对话历史
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})

        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []


# 初始化模型（首次运行需要下载，请耐心等待）
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
# 如显存不足，请使用：llm = QwenLLM(model_name="Qwen/Qwen2.5-3B-Instruct")

print("\n模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话")

# 03 - Agent 记忆系统：从短期记忆到长期记忆

## 学习目标

- 理解 Agent 记忆系统的层次结构
- 掌握短期记忆、长期记忆、工作记忆的实现方式
- 学习向量数据库在记忆系统中的应用
- 实现一个具备完整记忆能力的 Agent

---

## 1. 记忆系统概述

### 1.1 为什么 Agent 需要记忆？

人类智能的核心特征之一是**记忆**——我们能够：
- 记住刚才的对话内容（短期记忆）
- 回忆过去的经历（长期记忆）
- 在思考时保持相关信息（工作记忆）

Agent 同样需要记忆系统来：
- **保持对话连贯性**：记住之前的对话上下文
- **积累知识经验**：从交互中学习，避免重复错误
- **个性化服务**：记住用户偏好和习惯
- **复杂任务执行**：多步骤任务中保持状态

### 1.2 记忆系统的层次结构

```
┌─────────────────────────────────────────────────────────────┐
│                      记忆系统架构                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────┐    ┌─────────────────┐                │
│  │   短期记忆       │    │   长期记忆       │                │
│  │  (Short-term)   │    │  (Long-term)    │                │
│  │                 │    │                 │                │
│  │ • 对话历史      │    │ • 用户画像      │                │
│  │ • 当前上下文    │◄──►│ • 知识库        │                │
│  │ • 最近交互      │    │ • 经验总结      │                │
│  │                 │    │ • 技能记忆      │                │
│  │ 存储：内存      │    │ 存储：向量数据库 │                │
│  │ 时效：秒~分钟   │    │ 时效：永久      │                │
│  └────────┬────────┘    └────────┬────────┘                │
│           │                      │                         │
│           └──────────┬───────────┘                         │
│                      ▼                                      │
│           ┌─────────────────┐                              │
│           │    工作记忆      │                              │
│           │ (Working Memory)│                              │
│           │                 │                              │
│           │ • 当前任务状态  │                              │
│           │ • 中间计算结果  │                              │
│           │ • 活跃上下文    │                              │
│           └─────────────────┘                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 短期记忆（Short-term Memory）

### 2.1 对话历史管理

短期记忆最核心的形式是**对话历史**，即维护最近的交互记录。

**关键问题**：
- 保留多少轮对话？（上下文窗口限制）
- 如何压缩历史信息？（摘要技术）
- 如何处理超长对话？（滑动窗口、分层摘要）

In [ ]:
# 短期记忆实现：对话历史管理
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Message:
    """对话消息"""
    role: str  # 'user', 'assistant', 'system', 'tool'
    content: str
    timestamp: datetime = field(default_factory=datetime.now)
    metadata: Dict = field(default_factory=dict)

class ShortTermMemory:
    """
    短期记忆管理器
    
    负责维护对话历史，支持多种管理策略
    """
    
    def __init__(
        self,
        max_messages: int = 20,  # 最大保留消息数
        max_tokens: int = 4000,  # 最大 token 数（估算）
        enable_summarization: bool = True
    ):
        self.max_messages = max_messages
        self.max_tokens = max_tokens
        self.enable_summarization = enable_summarization
        self.messages: List[Message] = []
        self.summary: str = ""  # 历史摘要
    
    def add_message(self, role: str, content: str, metadata: Dict = None):
        """添加消息"""
        msg = Message(
            role=role,
            content=content,
            metadata=metadata or {}
        )
        self.messages.append(msg)
        
        # 检查是否需要清理
        self._maybe_cleanup()
    
    def _maybe_cleanup(self):
        """根据需要清理历史记录"""
        
        # 策略 1：超过消息数量限制
        if len(self.messages) > self.max_messages:
            if self.enable_summarization:
                # 对旧消息进行摘要
                old_messages = self.messages[:-self.max_messages//2]
                self._summarize(old_messages)
                # 保留较新的消息
                self.messages = self.messages[-self.max_messages//2:]
            else:
                # 直接删除最旧的消息
                self.messages = self.messages[-self.max_messages:]
    
    def _summarize(self, messages: List[Message]):
        """
        对历史消息进行摘要
        
        实际应用中应调用 LLM 进行摘要
        这里使用简单的规则模拟
        """
        topics = set()
        for msg in messages:
            # 简单提取关键词（实际应用中使用 NLP）
            if "天气" in msg.content:
                topics.add("天气查询")
            elif "计算" in msg.content:
                topics.add("数学计算")
            elif "搜索" in msg.content:
                topics.add("信息搜索")
        
        if topics:
            self.summary = f"历史对话涉及: {', '.join(topics)}"
    
    def get_context(self, include_summary: bool = True) -> List[Dict]:
        """
        获取当前上下文
        
        Returns:
            格式化的消息列表，可直接用于 LLM API
        """
        context = []
        
        # 添加系统提示（包含摘要）
        system_prompt = "你是一个智能助手。"
        if include_summary and self.summary:
            system_prompt += f"\n\n历史对话摘要: {self.summary}"
        
        context.append({"role": "system", "content": system_prompt})
        
        # 添加历史消息
        for msg in self.messages:
            context.append({
                "role": msg.role,
                "content": msg.content
            })
        
        return context
    
    def clear(self):
        """清空记忆"""
        self.messages = []
        self.summary = ""
    
    def __len__(self):
        return len(self.messages)
    
    def __repr__(self):
        return f"ShortTermMemory(messages={len(self)}, summary='{self.summary[:50]}...')"

# 创建短期记忆实例
stm = ShortTermMemory(max_messages=10)
print("✅ 短期记忆管理器初始化完成")
print(stm)

In [ ]:
# 测试短期记忆

# 模拟多轮对话
conversations = [
    ("user", "你好，我想查询天气"),
    ("assistant", "您好！我可以帮您查询天气。请告诉我您想查询哪个城市？"),
    ("user", "北京"),
    ("assistant", "北京今天天气晴朗，25°C。还有其他需要吗？"),
    ("user", "帮我计算 15 * 23"),
    ("assistant", "15 * 23 = 345"),
    ("user", "再帮我查一下上海"),
    ("assistant", "上海今天多云，28°C。"),
]

for role, content in conversations:
    stm.add_message(role, content)
    print(f"添加: [{role}] {content[:30]}...")

print(f"\n当前记忆状态: {stm}")
print(f"消息数量: {len(stm)}")
print("\n上下文内容:")
for msg in stm.get_context():
    print(f"  [{msg['role']}] {msg['content'][:50]}...")

---

## 3. 长期记忆（Long-term Memory）

### 3.1 向量数据库记忆

长期记忆通常使用**向量数据库**实现，通过 Embedding 将文本转换为向量进行存储和检索。

**核心流程**：
1. **存储**：文本 → Embedding 模型 → 向量 → 存入向量数据库
2. **检索**：查询文本 → Embedding 模型 → 向量 → 相似度搜索 → 返回相关记忆

**常用向量数据库**：
- **Chroma**：轻量级，适合本地开发
- **Milvus**：企业级，支持大规模数据
- **Pinecone**：托管服务，易用性强
- **Weaviate**：支持 GraphQL 查询
- **Qdrant**：Rust 编写，高性能

### 3.2 实现基于向量的长期记忆

In [ ]:
# 长期记忆实现：基于简单向量相似度的记忆系统
# 实际应用中应使用 Chroma、Milvus 等向量数据库

import numpy as np
from typing import List, Tuple
import hashlib

class SimpleEmbedding:
    """
    简化版 Embedding 模型
    
    实际应用中应使用：
    - OpenAI: text-embedding-ada-002
    - 智谱 AI: embedding-2
    - 通义千问: text-embedding-v1
    """
    
    def __init__(self, dim: int = 128):
        self.dim = dim
    
    def encode(self, text: str) -> np.ndarray:
        """
        将文本编码为向量
        
        这里使用简单的哈希方法模拟
        实际应用中使用预训练模型
        """
        # 使用文本哈希生成确定性向量
        hash_val = hashlib.md5(text.encode()).hexdigest()
        np.random.seed(int(hash_val[:8], 16))
        vector = np.random.randn(self.dim)
        # 归一化
        vector = vector / np.linalg.norm(vector)
        return vector

class LongTermMemory:
    """
    长期记忆管理器
    
    基于向量相似度检索的记忆系统
    """
    
    def __init__(self, embedding_dim: int = 128):
        self.embedding = SimpleEmbedding(dim=embedding_dim)
        self.memories: List[Dict] = []  # 存储记忆和向量
        self.embedding_dim = embedding_dim
    
    def add_memory(
        self,
        content: str,
        memory_type: str = "fact",  # fact, experience, preference
        metadata: Dict = None
    ):
        """
        添加长期记忆
        
        Args:
            content: 记忆内容
            memory_type: 记忆类型 (fact/experience/preference)
            metadata: 额外元数据
        """
        vector = self.embedding.encode(content)
        
        memory = {
            "id": len(self.memories),
            "content": content,
            "type": memory_type,
            "vector": vector,
            "timestamp": datetime.now(),
            "metadata": metadata or {},
            "access_count": 0  # 访问计数，用于记忆强化
        }
        
        self.memories.append(memory)
        print(f"[记忆存储] 类型={memory_type}, 内容={content[:50]}...")
    
    def retrieve(
        self,
        query: str,
        top_k: int = 3,
        memory_type: str = None
    ) -> List[Dict]:
        """
        检索相关记忆
        
        Args:
            query: 查询文本
            top_k: 返回最相关的 k 条记忆
            memory_type: 按类型过滤
        """
        if not self.memories:
            return []
        
        query_vector = self.embedding.encode(query)
        
        # 计算相似度
        similarities = []
        for memory in self.memories:
            # 类型过滤
            if memory_type and memory["type"] != memory_type:
                continue
            
            # 计算余弦相似度
            similarity = np.dot(query_vector, memory["vector"])
            similarities.append((similarity, memory))
        
        # 按相似度排序
        similarities.sort(reverse=True, key=lambda x: x[0])
        
        # 返回 top_k
        results = []
        for sim, memory in similarities[:top_k]:
            memory["access_count"] += 1
            results.append({
                "content": memory["content"],
                "type": memory["type"],
                "similarity": float(sim),
                "metadata": memory["metadata"]
            })
        
        return results
    
    def get_user_profile(self) -> Dict:
        """获取用户画像（基于 preference 类型记忆）"""
        preferences = [
            m for m in self.memories
            if m["type"] == "preference"
        ]
        return {
            "preference_count": len(preferences),
            "preferences": [p["content"] for p in preferences]
        }
    
    def forget(self, memory_id: int):
        """删除特定记忆"""
        if 0 <= memory_id < len(self.memories):
            del self.memories[memory_id]
            # 重新编号
            for i, mem in enumerate(self.memories):
                mem["id"] = i
    
    def __len__(self):
        return len(self.memories)

# 创建长期记忆实例
ltm = LongTermMemory()
print("✅ 长期记忆管理器初始化完成")

In [ ]:
# 添加示例记忆

# 事实类记忆
ltm.add_memory(
    content="用户张三喜欢 Python 编程，常用 Django 框架",
    memory_type="fact",
    metadata={"source": "user_profile", "confidence": 0.9}
)

ltm.add_memory(
    content="用户张三在北京工作，从事软件开发",
    memory_type="fact",
    metadata={"source": "conversation", "confidence": 0.8}
)

# 偏好类记忆
ltm.add_memory(
    content="用户喜欢简洁的回答，不喜欢冗长解释",
    memory_type="preference",
    metadata={"source": "feedback", "confidence": 0.95}
)

ltm.add_memory(
    content="用户对机器学习话题感兴趣",
    memory_type="preference",
    metadata={"source": "interaction_history"}
)

# 经验类记忆
ltm.add_memory(
    content="上次帮助用户解决 Django 数据库迁移问题，需要执行 makemigrations 和 migrate",
    memory_type="experience",
    metadata={"task": "debug", "success": True}
)

print(f"\n总共存储了 {len(ltm)} 条记忆")

In [ ]:
# 测试记忆检索

# 查询 1：与编程相关
print("查询: 用户喜欢什么编程语言？")
results = ltm.retrieve("用户喜欢什么编程语言", top_k=2)
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['type']}] {r['content']}")
    print(f"     相似度: {r['similarity']:.3f}")

print("\n" + "="*50)

# 查询 2：与偏好相关
print("查询: 用户喜欢什么样的回答风格？")
results = ltm.retrieve("用户喜欢简洁回答", top_k=2, memory_type="preference")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['type']}] {r['content']}")
    print(f"     相似度: {r['similarity']:.3f}")

print("\n" + "="*50)

# 查询 3：获取用户画像
print("用户画像:")
profile = ltm.get_user_profile()
print(f"  偏好数量: {profile['preference_count']}")
for pref in profile['preferences']:
    print(f"  - {pref}")

---

## 4. 工作记忆（Working Memory）

### 4.1 工作记忆的作用

工作记忆是 Agent 在执行任务时的**临时工作空间**，用于：
- 保存当前任务的目标和进度
- 存储中间计算结果
- 维护当前活跃的上下文信息

### 4.2 实现工作记忆

In [ ]:
class WorkingMemory:
    """
    工作记忆管理器
    
    维护当前任务的临时状态
    """
    
    def __init__(self):
        self.current_task: Optional[str] = None
        self.task_progress: Dict = {}
        self.intermediate_results: List[Dict] = []
        self.active_context: Dict = {}
    
    def set_task(self, task: str, steps: List[str]):
        """设置当前任务"""
        self.current_task = task
        self.task_progress = {
            "total_steps": len(steps),
            "completed_steps": [],
            "remaining_steps": steps.copy(),
            "current_step": None
        }
        print(f"[工作记忆] 设置任务: {task}")
        print(f"[工作记忆] 任务步骤: {steps}")
    
    def complete_step(self, step: str, result: Any):
        """完成一个步骤"""
        if step in self.task_progress["remaining_steps"]:
            self.task_progress["remaining_steps"].remove(step)
            self.task_progress["completed_steps"].append(step)
            
            self.intermediate_results.append({
                "step": step,
                "result": result,
                "timestamp": datetime.now()
            })
            
            print(f"[工作记忆] 完成步骤: {step}")
            print(f"[工作记忆] 结果: {result}")
    
    def set_context(self, key: str, value: Any):
        """设置上下文变量"""
        self.active_context[key] = value
    
    def get_context(self, key: str) -> Any:
        """获取上下文变量"""
        return self.active_context.get(key)
    
    def get_progress(self) -> Dict:
        """获取任务进度"""
        total = self.task_progress["total_steps"]
        completed = len(self.task_progress["completed_steps"])
        return {
            "task": self.current_task,
            "progress": f"{completed}/{total}",
            "percentage": (completed / total * 100) if total > 0 else 0,
            "current_step": self.task_progress["current_step"],
            "completed": self.task_progress["completed_steps"],
            "remaining": self.task_progress["remaining_steps"]
        }
    
    def clear(self):
        """清空工作记忆"""
        self.current_task = None
        self.task_progress = {}
        self.intermediate_results = []
        self.active_context = {}
    
    def __repr__(self):
        if self.current_task:
            progress = self.get_progress()
            return f"WorkingMemory(task='{self.current_task}', progress={progress['progress']})"
        return "WorkingMemory(empty)"

# 创建工作记忆实例
wm = WorkingMemory()
print("✅ 工作记忆管理器初始化完成")

In [ ]:
# 测试工作记忆

# 设置一个复杂任务
wm.set_task(
    task="帮用户预订从北京到上海的机票",
    steps=[
        "查询航班信息",
        "比较价格和时刻",
        "确认用户选择",
        "填写乘客信息",
        "完成支付"
    ]
)

# 设置上下文
wm.set_context("departure_city", "北京")
wm.set_context("arrival_city", "上海")
wm.set_context("date", "2025-01-15")

print(f"\n{wm}")

# 完成第一步
wm.complete_step(
    "查询航班信息",
    {"flights": ["CA1234", "MU5678"], "prices": [1200, 980]}
)

# 查看进度
print("\n当前进度:")
progress = wm.get_progress()
for key, value in progress.items():
    print(f"  {key}: {value}")

# 查看中间结果
print("\n中间结果:")
for result in wm.intermediate_results:
    print(f"  {result['step']}: {result['result']}")

---

## 5. 完整记忆系统集成

将短期记忆、长期记忆、工作记忆整合到一个统一的 Agent 中。

In [ ]:
# ============================================================
# 具备完整记忆系统的 Agent（已集成 QwenLLM）
# ============================================================

class MemoryEnabledAgent:
    """
    具备完整记忆系统的 Agent
    
    集成：
    - 短期记忆：对话历史
    - 长期记忆：知识库存储
    - 工作记忆：任务状态
    """
    
    def __init__(self, name: str = "MemoryAgent"):
        self.name = name
        self.short_term = ShortTermMemory(max_messages=10)
        self.long_term = LongTermMemory()
        self.working = WorkingMemory()
        
        # 从长期记忆加载用户画像到短期记忆
        self._load_user_profile()
    
    def _load_user_profile(self):
        """加载用户画像到系统提示"""
        profile = self.long_term.get_user_profile()
        if profile["preference_count"] > 0:
            prefs = "\n".join([f"- {p}" for p in profile["preferences"]])
            self.short_term.add_message(
                "system",
                f"用户偏好：\n{prefs}"
            )
    
    def process(self, user_input: str) -> str:
        """
        处理用户输入
        
        流程：
        1. 检索长期记忆获取相关背景
        2. 将用户输入加入短期记忆
        3. 根据上下文生成回答
        4. 将回答加入短期记忆
        5. 提取重要信息存入长期记忆
        """
        
        print(f"\n{'='*60}")
        print(f"🚀 Agent '{self.name}' 处理输入")
        print(f"{'='*60}")
        
        # Step 1: 检索长期记忆
        print("\n📚 [长期记忆检索]")
        relevant_memories = self.long_term.retrieve(user_input, top_k=2)
        if relevant_memories:
            print(f"找到 {len(relevant_memories)} 条相关记忆:")
            for mem in relevant_memories:
                print(f"  - [{mem['type']}] {mem['content'][:50]}...")
        else:
            print("未找到相关记忆")
        
        # Step 2: 添加用户输入到短期记忆
        self.short_term.add_message("user", user_input)
        
        # Step 3: 生成回答（模拟）
        print("\n🤔 [生成回答]")
        
        # 结合记忆生成回答
        context_parts = ["基于您的提问:"]
        
        if relevant_memories:
            context_parts.append("我回忆起以下相关信息:")
            for mem in relevant_memories:
                context_parts.append(f"- {mem['content']}")
        
        context_parts.append(f"\n关于您的问题 '{user_input}'，我的回答是:")
        
        # ---- 无模型时的备选方案（已注释）----
        # if "天气" in user_input:
        #     answer = "北京今天天气晴朗，25°C。根据您的位置偏好，我推荐您关注空气质量。"
        # elif "编程" in user_input or "代码" in user_input:
        #     answer = "作为 Python 爱好者，我建议您使用 Django 框架进行 Web 开发。"
        # else:
        #     answer = f"我理解了您关于 '{user_input}' 的问题。让我为您提供帮助。"
        # ---- 备选方案结束 ----

        # 使用真实 Qwen 模型生成回答
        memory_context = ""
        if relevant_memories:
            memory_context = "\n".join([f"- {mem['content']}" for mem in relevant_memories])

        try:
            answer = llm.chat(
                user_input,
                system_prompt=f"你是一个智能助手。以下是相关记忆信息：\n{memory_context}\n请结合记忆信息回答用户问题。",
                max_new_tokens=256
            )
        except Exception as e:
            answer = f"回答生成失败: {str(e)}"
        
        context_parts.append(answer)
        final_answer = "\n".join(context_parts)
        
        # Step 4: 添加回答到短期记忆
        self.short_term.add_message("assistant", final_answer)
        
        # Step 5: 提取信息存入长期记忆（简化版）
        self._extract_and_store(user_input, final_answer)
        
        print(f"\n📤 [最终回答]")
        print(final_answer)
        
        return final_answer
    
    def _extract_and_store(self, user_input: str, answer: str):
        """
        从交互中提取重要信息存入长期记忆
        
        实际应用中应使用 LLM 进行信息提取
        """
        # 简单规则：如果用户表达了偏好，存入长期记忆
        preference_keywords = ["喜欢", "讨厌", "偏好", "想要", "不需要"]
        if any(kw in user_input for kw in preference_keywords):
            self.long_term.add_memory(
                content=f"用户表达: {user_input}",
                memory_type="preference"
            )
            print("\n💾 [长期记忆] 已保存用户偏好")
    
    def get_memory_status(self) -> Dict:
        """获取记忆系统状态"""
        return {
            "short_term_messages": len(self.short_term),
            "long_term_memories": len(self.long_term),
            "current_task": self.working.current_task,
            "user_profile": self.long_term.get_user_profile()
        }

# 创建具备完整记忆的 Agent
agent = MemoryEnabledAgent(name="智能助手")
print("✅ 具备完整记忆系统的 Agent 初始化完成")

In [ ]:
# 预加载一些长期记忆
agent.long_term.add_memory(
    content="用户喜欢 Python 编程，常用 Django 框架",
    memory_type="fact"
)

agent.long_term.add_memory(
    content="用户偏好简洁的回答风格",
    memory_type="preference"
)

# 测试交互
agent.process("我想学习 Web 开发，有什么建议？")

In [ ]:
# 继续对话
agent.process("今天北京的天气怎么样？")

In [ ]:
# 查看记忆状态
status = agent.get_memory_status()
print("Agent 记忆系统状态:")
for key, value in status.items():
    print(f"  {key}: {value}")

---

## 6. 记忆系统的设计要点

### 6.1 记忆存储策略

| 策略 | 描述 | 适用场景 |
|------|------|----------|
| **全部存储** | 保存所有交互 | 数据量小、重要性高的场景 |
| **摘要存储** | 定期摘要压缩 | 长对话、历史积累场景 |
| **选择性存储** | 只存储重要信息 | 资源受限、噪声多的场景 |
| **分层存储** | 不同时间粒度 | 复杂 Agent 系统 |

### 6.2 记忆检索策略

| 策略 | 描述 | 优点 |
|------|------|------|
| **向量相似度** | 语义检索 | 理解同义词、相关概念 |
| **关键词匹配** | 精确检索 | 速度快、确定性高 |
| **混合检索** | 向量 + 关键词 | 兼顾语义和精确性 |
| **时间衰减** | 近期记忆优先 | 关注当前上下文 |

### 6.3 记忆遗忘机制

```python
# 基于访问频率的遗忘
def should_forget(memory, current_time):
    # 访问次数越少，越容易遗忘
    # 时间越久，越容易遗忘
    age = current_time - memory['timestamp']
    access_score = memory['access_count'] / (age.days + 1)
    return access_score < threshold

# 基于重要性的遗忘
def should_forget_by_importance(memory):
    # 用户明确标记重要的不遗忘
    if memory.get('important', False):
        return False
    # 偏好类记忆保留更久
    if memory['type'] == 'preference':
        return False
    return True
```

---

## 7. 小结

### 核心要点

1. **三层记忆架构**：短期记忆（对话历史）+ 长期记忆（向量数据库）+ 工作记忆（任务状态）
2. **短期记忆**：维护最近交互，支持摘要压缩
3. **长期记忆**：基于向量相似度检索，持久化存储
4. **工作记忆**：管理当前任务状态和中间结果
5. **记忆策略**：存储策略、检索策略、遗忘机制的综合设计

### 下一步

- [04_planning_and_reflection.ipynb](04_planning_and_reflection.ipynb) - 学习规划与反思模式
- [02_frameworks/00_langchain_basics.ipynb](../02_frameworks/00_langchain_basics.ipynb) - 使用 LangChain 实现记忆系统

---

## 参考资源

- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560)
- [Chroma DB 文档](https://docs.trychroma.com/)
- [Milvus 文档](https://milvus.io/docs)
- [LangChain Memory 模块](https://python.langchain.com/docs/modules/memory/)